<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/ResNet18_entrenada_contra_gini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json

import kagglehub
path4 = kagglehub.dataset_download("leonardocaravaggio/ge-images4")
path5 = kagglehub.dataset_download("leonardocaravaggio/ge-images5")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 2.89G/2.89G [00:46<00:00, 67.4MB/s]

Extracting files...


100%|██████████| 895M/895M [00:11<00:00, 80.0MB/s]

Extracting files...


In [ ]:
from PIL import Image
import torch
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # convierte a tensor (C,H,W) y escala a [0,1]
])

def cargar_imagenes_multiescala(nombre_base):
    escalas = ["1K", "5K", "10K", "15K"]
    imagenes = []

    for escala in escalas:
        path = f"imagenes/{nombre_base} - {escala}.png"
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = transform(img)
            imagenes.append(img_tensor)
        except Exception as e:
            print(f"Error con {escala}: {e}")
            return None  # o podés devolver un tensor de ceros

    # Concatenar en el eje de canales: (3,224,224) * 4 => (12,224,224)
    return torch.cat(imagenes, dim=0)


In [4]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
from tqdm import tqdm

class GiniMultiEscalaDataset(Dataset):
    def __init__(self, df, path1, path2, transform=None):
        self.df = df.copy()
        self.path1 = path1
        self.path2 = path2
        self.transform = transform
        self.sufijos = [" - 1K.png", " - 5K.png", " - 10K.png", " - 15K.png"]

    def sanear_nombre_ciudad(self, nombre):
        return nombre.replace("/", ".").replace(":", "_").replace("'", "!").replace("\\", "").strip()

    def cargar_multiimagen(self, ciudad):
        nombre = self.sanear_nombre_ciudad(ciudad)
        imgs = []
        for suf in self.sufijos:
            img_path = None
            for base_path in [self.path1, self.path2]:
                path = os.path.join(base_path, f"{nombre}{suf}")
                if os.path.exists(path):
                    img_path = path
                    break
            if img_path is None:
                print(f"⚠️ No se encontró imagen para {ciudad} - {suf}")
                img = Image.new("RGB", (512, 512), (0, 0, 0))  # imagen negra
            else:
                img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            imgs.append(img)
        return torch.cat(imgs, dim=0)  # (12, 512, 512)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ciudad = row["Ciudad"]
        gini = row["Gini"]
        image = self.cargar_multiimagen(ciudad)
        label = torch.tensor(gini, dtype=torch.float32)
        return image, label

    def __len__(self):
        return len(self.df)

In [32]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
# Modificamos la primera capa
mobilenet.features[0][0] = nn.Conv2d(12, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Freeze o no freeze, según prefieras
for param in mobilenet.parameters():
    param.requires_grad = True

model = nn.Sequential(
    mobilenet.features,
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten(),
    nn.Linear(1280, 64),
    nn.ReLU(),
    nn.Dropout(p=0.4),  # Dropout
    nn.Linear(64, 1)
).to(DEVICE)


In [33]:
# ==== PARÁMETROS ====
IMG_TYPE = "10K"  # Puede ser "1K", "5K", "10K", "15K"
BATCH_SIZE = 8
EPOCHS = 12
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHT_DECAY=1e-5

# ==== CARGA ====
df_eph = pd.read_csv("Gini_EPH.csv")
df_ocde = pd.read_csv(path4 + "/Gini con latlon.csv")

# ==== NORMALIZAR COLUMNAS ====
# Renombrar columnas para que coincidan
df_eph = df_eph.rename(columns={"Nombre_Aglomerado": "Ciudad", "Gini_Hogares": "Gini"})
df_ocde["Gini"] = df_ocde["Gini"].str.replace(',', '.', regex=False).astype(float)

# ==== CONCATENAR ====
df_merged = pd.concat([df_eph[["Ciudad", "Gini"]], df_ocde[["Ciudad", "Gini"]]], ignore_index=True)

# ==== TRANSFORMACIONES ====
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ==== CARGA DE DATOS ====
train_df, val_df = train_test_split(df_merged, test_size=0.2, random_state=42)
train_dataset = GiniMultiEscalaDataset(train_df, path4, path5, transform)
val_dataset = GiniMultiEscalaDataset(val_df, path4, path5, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

train_dataset = GiniMultiEscalaDataset(train_df, path4, path5, transform)
val_dataset = GiniMultiEscalaDataset(val_df, path4, path5, transform)

In [34]:
# ==== OPTIMIZADOR Y PÉRDIDA ====
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==== ENTRENAMIENTO ====
best_r_val = -1
patience = 3
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")

    # VALIDACIÓN
    model.eval()
    preds_val, trues_val = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs).cpu().numpy().flatten()
            preds_val.extend(outputs)
            trues_val.extend(targets.numpy())
    r_val, p_val = pearsonr(preds_val, trues_val)
    print(f"📊 Pearson Val: {r_val:.3f} | p-val: {p_val:.5f}")

    # EARLY STOPPING
    if r_val > best_r_val:
        best_r_val = r_val
        patience_counter = 0
        torch.save(model.state_dict(), "mejor_modelo.pt")
        print("✅ Nuevo mejor modelo guardado.")
    else:
        patience_counter += 1
        print(f"⚠️ Sin mejora. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("⛔ Early stopping activado.")
            break


model.load_state_dict(torch.load("mejor_modelo.pt"))

# ==== VALIDACIÓN ====
model.eval()
preds_val, trues_val = [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_val.extend(outputs)
        trues_val.extend(targets.numpy())

r_val, p_val = pearsonr(preds_val, trues_val)

# Evaluar también en entrenamiento para referencia
preds_train, trues_train = [], []
with torch.no_grad():
    for inputs, targets in train_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_train.extend(outputs)
        trues_train.extend(targets.numpy())

r_train, p_train = pearsonr(preds_train, trues_train)

print(f"\n📊 Pearson Train: {r_train:.3f} | p-value Train: {p_train:.5f}")
print(f"📊 Pearson Val: {r_val:.3f} | p-value Val: {p_val:.5f}")

Epoch 1/12: 100%|██████████| 15/15 [03:47<00:00, 15.14s/it]


Train Loss: 0.1154
📊 Pearson Val: 0.163 | p-val: 0.38814
✅ Nuevo mejor modelo guardado.


Epoch 2/12: 100%|██████████| 15/15 [03:51<00:00, 15.44s/it]


Train Loss: 0.0155
📊 Pearson Val: 0.186 | p-val: 0.32503
✅ Nuevo mejor modelo guardado.


Epoch 3/12: 100%|██████████| 15/15 [03:45<00:00, 15.01s/it]


...

Train Loss: 0.0112
📊 Pearson Val: 0.399 | p-val: 0.02885
⚠️ Sin mejora. Patience: 1/3


Epoch 5/12: 100%|██████████| 15/15 [03:58<00:00, 15.89s/it]


Train Loss: 0.0096
📊 Pearson Val: 0.392 | p-val: 0.03210
⚠️ Sin mejora. Patience: 2/3


Epoch 6/12: 100%|██████████| 15/15 [03:45<00:00, 15.02s/it]


Train Loss: 0.0113
📊 Pearson Val: 0.406 | p-val: 0.02614
⚠️ Sin mejora. Patience: 3/3
⛔ Early stopping activado.

📊 Pearson Train: 0.475 | p-value Train: 0.00000
📊 Pearson Val: 0.460 | p-value Val: 0.01052


# Validación

In [22]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path3 = kagglehub.dataset_download("leonardocaravaggio/ge-images3")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 336M/336M [00:03<00:00, 89.8MB/s]

Extracting files...


In [35]:
from PIL import Image
import torch
import os
from torchvision import transforms

# Asegurate de que tu modelo esté en modo evaluación
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms (igual que en entrenamiento)
transform = transforms.Compose([
    transforms.Resize((512, 512)),  # asegurate que sea igual al entrenamiento
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Lista de ciudades
lista_ciudades = [
    "Desierto",  "Amazonas", "Oceano", "Santiago de Cuba", "Curitiba", "El Alto",
    "Montevideo", "Lo Barnechea", "Los Cedros", "La Cava", "Rocinha", "Retiro"
]

for nombre in lista_ciudades:
    nombre_archivo = nombre.replace("/", ".").replace(":", "_").replace("'", "!").replace("\\", "").strip()
    ruta_base = os.path.join(path3, nombre_archivo)

    imgs = []
    escalas_disponibles = []

    for escala in ["1K", "5K", "10K", "15K"]:
        path_img = f"{ruta_base} - {escala}.png"
        try:
            img = Image.open(path_img).convert("RGB")
            img_tensor = transform(img)
            imgs.append(img_tensor)
            escalas_disponibles.append(escala)
        except Exception as e:
            print(f"{nombre} - {escala}: ❌ Imagen no cargada ({e})")

    if imgs:
        input_concat = torch.cat(imgs, dim=0).unsqueeze(0).to(device)  # (1, 12, H, W)
        with torch.no_grad():
            pred = model(input_concat).item()
        print(f"\n📍 {nombre}:")
        print(f"  🎯 Desigualdad: {pred:.2f}")

    else:
        print(f"\n{name}: ❌ No se pudo procesar ninguna imagen.")



📍 Desierto:
  🎯 Desigualdad: 0.30

📍 Amazonas:
  🎯 Desigualdad: 0.32

📍 Oceano:
  🎯 Desigualdad: 0.04

📍 Santiago de Cuba:
  🎯 Desigualdad: 0.28

📍 Curitiba:
  🎯 Desigualdad: 0.20

📍 El Alto:
  🎯 Desigualdad: 0.39

📍 Montevideo:
  🎯 Desigualdad: 0.38

📍 Lo Barnechea:
  🎯 Desigualdad: 0.20

📍 Los Cedros:
  🎯 Desigualdad: 0.24

📍 La Cava:
  🎯 Desigualdad: 0.26

📍 Rocinha:
  🎯 Desigualdad: 0.28

📍 Retiro:
  🎯 Desigualdad: 0.31


# Inferencia

In [25]:
path1 = kagglehub.dataset_download("leonardocaravaggio/ge-images")
path2 = kagglehub.dataset_download("leonardocaravaggio/ge-images2")

100%|██████████| 13.3G/13.3G [04:37<00:00, 51.3MB/s]

Extracting files...


100%|██████████| 13.9G/13.9G [05:11<00:00, 48.1MB/s]

Extracting files...


In [28]:
import os
import shutil

# Crear una carpeta de destino
dest_folder = "imagenes"
os.makedirs(dest_folder, exist_ok=True)

# Función para copiar imágenes a una sola carpeta
def mover_imagenes(origen, destino):
    for root, _, files in os.walk(origen):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                shutil.move(os.path.join(root, file), os.path.join(destino, file))

# Copiar imágenes de ambos datasets al mismo folder
mover_imagenes(path1, dest_folder)
mover_imagenes(path2, dest_folder)

print(f"Imágenes combinadas en la carpeta: {dest_folder}")

Imágenes combinadas en la carpeta: imagenes


In [37]:
import pandas as pd
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
import torch
from torchvision import transforms

# Configuración
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Transformaciones
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet
                         std=[0.229, 0.224, 0.225])
])

# Cargar base
ciudades = pd.read_csv("base.csv")

# Asegurar columnas
for km in ["1km", "5km", "10km", "15km"]:
    ciudades[f'Desigualdad_{km}'] = np.nan

# Procesamiento por ciudad
for idx, row in tqdm(ciudades.iterrows(), total=len(ciudades)):
    nombre = row['City']
    nombre_archivo = nombre.replace("/", ".").replace(":", "_").replace("'", "!")
    ruta_completa = os.path.join('imagenes', nombre_archivo)

    imgs = []
    escalas = ["1K", "5K", "10K", "15K"]

    for escala in escalas:
        path_img = f"{ruta_completa} - {escala}.png"
        try:
            img = Image.open(path_img).convert("RGB")
            img = transform(img)
            imgs.append(img)
        except Exception as e:
            print(f"{nombre} - {escala}: ⚠️ Error al abrir imagen: {e}")
            imgs = []  # invalidamos la entrada
            break

    if len(imgs) == 4:
        input_concat = torch.cat(imgs, dim=0).unsqueeze(0).to(device)  # (1, 12, H, W)
        with torch.no_grad():
            pred = model(input_concat).item()

        ciudades.at[idx, f'Desigualdad'] = pred
    else:
        ciudades.at[idx, f'Desigualdad'] = np.nan

# Guardar
ciudades.to_csv("base_con_desigualdad.csv", index=False)
print("✅ Predicciones guardadas en 'base_con_desigualdad.csv'")

100%|██████████| 1095/1095 [20:30<00:00,  1.12s/it]

✅ Predicciones guardadas en 'base_con_desigualdad.csv'


In [39]:
from scipy.stats import pearsonr

# Eliminar pares con NaN
x = ciudades["Desigualdad"]
y = ciudades["P1ST"]
mask = x.notna() & y.notna()

# Calcular correlación de Pearson y p-value
r, p = pearsonr(x[mask], y[mask])

print(f"Coeficiente de Pearson: {r:.3f}")
print(f"Valor p: {p:.5f}")

Coeficiente de Pearson: -0.036
Valor p: 0.23924
